# Kasus A — Deteksi Penipuan Rekening Bank (BAF, varian Base, month 0–3)

**Tujuan:** tangkap penipu sebanyak mungkin, nasabah jujur yang tertandai maksimal 5% (FPR ≤ 5%).
**Metrik utama:** TPR pada FPR 5%. Pelengkap: ROC-AUC dan PR-AUC.

**Pembagian data (temporal, sesuai sifat data BAF yang berurutan per bulan):**
- Latih: month 0–1
- Validasi: month 2 (pilih model, tuning, dan tentukan threshold)
- Uji: month 3 (dipakai satu kali di akhir)

Alasan: di produksi model dilatih dari masa lalu dan dipakai pada bulan berikutnya. Split acak akan membocorkan pola waktu.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
MAX_FPR = 0.05
TARGET = "fraud_bool"
DATA_PATH = Path("data/Base.csv")
MODEL_PATH = Path("model/model_fraud.joblib")

# Kolom yang memakai -1 sebagai 'nilai hilang'. Dicek dari data: -1 hanya muncul
# di kolom ini dan tidak ada NaN di file. Kolom lain bernilai negatif sungguhan.
MISSING_MINUS1 = [
    "prev_address_months_count", "current_address_months_count", "bank_months_count",
    "session_length_in_minutes", "device_distinct_emails_8w",
]

## 1. Muat data (per potongan) dan bagi

In [2]:
# File 1 juta baris dibaca per potongan; hanya month 0-3 yang disimpan.
df = pd.concat(chunk[chunk["month"] <= 3] for chunk in pd.read_csv(DATA_PATH, chunksize=200_000))

# month dipakai untuk membagi data, bukan sebagai fitur.
# device_fraud_count bernilai konstan 0 di data ini, jadi dibuang.
X = df.drop(columns=[TARGET, "month", "device_fraud_count"])
y = df[TARGET]

split = {"latih": df["month"] <= 1, "validasi": df["month"] == 2, "uji": df["month"] == 3}
X_train, y_train = X[split["latih"]], y[split["latih"]]
X_val, y_val = X[split["validasi"]], y[split["validasi"]]
X_test, y_test = X[split["uji"]], y[split["uji"]]

pd.DataFrame({k: {"baris": m.sum(), "penipu": y[m].sum(), "rasio penipu (%)": round(100 * y[m].mean(), 2)}
              for k, m in split.items()}).T

,baris,penipu,rasio penipu (%)
latih,260060.0,2698.0,1.04
validasi,136979.0,1198.0,0.87
uji,150936.0,1392.0,0.92


## 2. Preprocessing dan fungsi evaluasi

- Semua preprocessing ada di dalam `Pipeline` (di-fit hanya pada data latih, tidak ada leakage).
- `-1` pada 5 kolom diimputasi median, dengan kolom indikator 'hilang' (ketiadaan data bisa jadi sinyal penipuan).
- **Class imbalance:** `class_weight="balanced"`. Alasan: penipu hanya ~1%. Bobot menaikkan pengaruh penipu tanpa membuang data atau membuat baris sintetis. Threshold ditentukan ulang dari FPR 5% di validasi, jadi yang dinilai adalah kualitas peringkat skor.

In [3]:
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = [c for c in X.select_dtypes(include="number").columns if c not in MISSING_MINUS1]


def build_pipeline(model):
    prep = ColumnTransformer([
        ("hilang", make_pipeline(SimpleImputer(missing_values=-1, strategy="median", add_indicator=True),
                                 StandardScaler()), MISSING_MINUS1),
        ("angka", StandardScaler(), num_cols),
        ("kategori", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    return Pipeline([("prep", prep), ("model", model)])


def evaluate(pipe, X, y, threshold=None):
    """TPR/FPR pada threshold + ROC-AUC + PR-AUC.
    threshold=None: threshold = skor terendah dengan FPR <= 5% pada data ini (dipakai di validasi).
    threshold diisi: pakai threshold itu (dipakai di data uji)."""
    proba = pipe.predict_proba(X)[:, 1]
    if threshold is None:
        fpr, _, thresholds = roc_curve(y, proba)
        threshold = thresholds[np.flatnonzero(fpr <= MAX_FPR)[-1]]
    flagged = proba >= threshold
    return {"TPR": flagged[y == 1].mean(), "FPR": flagged[y == 0].mean(),
            "ROC-AUC": roc_auc_score(y, proba), "PR-AUC": average_precision_score(y, proba),
            "threshold": threshold}

## 3. Baseline: Logistic Regression

In [4]:
scores = {}
lr = build_pipeline(LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)).fit(X_train, y_train)
scores["LogisticRegression"] = evaluate(lr, X_val, y_val)
pd.DataFrame(scores).T

,TPR,FPR,ROC-AUC,PR-AUC,threshold
LogisticRegression,0.492487,0.049882,0.875751,0.114693,0.789408


## 4. Tree-based: HistGradientBoosting + tuning
Grid kecil, dinilai dengan TPR pada FPR 5% di **validasi** (data uji tidak disentuh).

In [5]:
grid = ParameterGrid({"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31], "l2_regularization": [0.0, 1.0]})
tuning = []
for params in grid:
    hgb = build_pipeline(HistGradientBoostingClassifier(
        class_weight="balanced", max_iter=300, early_stopping=True, random_state=SEED, **params)).fit(X_train, y_train)
    tuning.append({**params, **evaluate(hgb, X_val, y_val)})
tuning = pd.DataFrame(tuning).sort_values("TPR", ascending=False).reset_index(drop=True)
tuning

,l2_regularization,learning_rate,max_leaf_nodes,TPR,FPR,ROC-AUC,PR-AUC,threshold
0,0.0,0.05,15,0.540902,0.050000,0.885133,0.131598,0.740304
1,1.0,0.05,31,0.540067,0.050000,0.883321,0.129407,0.731043
2,1.0,0.05,15,0.539232,0.049838,0.884059,0.131570,0.739765
3,0.0,0.10,15,0.539232,0.049852,0.884116,0.129004,0.739417
4,1.0,0.10,15,0.539232,0.049948,0.884075,0.129502,0.741595
5,0.0,0.05,31,0.536728,0.049919,0.884536,0.133925,0.729632
6,1.0,0.10,31,0.535058,0.049698,0.882106,0.130434,0.732858
7,0.0,0.10,31,0.523372,0.049985,0.881586,0.127174,0.733821


In [6]:
best = tuning.iloc[0][list(grid[0])].to_dict()
best["max_leaf_nodes"] = int(best["max_leaf_nodes"])
hgb = build_pipeline(HistGradientBoostingClassifier(
    class_weight="balanced", max_iter=300, early_stopping=True, random_state=SEED, **best)).fit(X_train, y_train)
scores["HistGradientBoosting (tuned)"] = evaluate(hgb, X_val, y_val)

val_table = pd.DataFrame(scores).T.round(4)
print("Skor VALIDASI (month 2):")
val_table

Skor VALIDASI (month 2):


,TPR,FPR,ROC-AUC,PR-AUC,threshold
LogisticRegression,0.4925,0.0499,0.8758,0.1147,0.7894
HistGradientBoosting (tuned),0.5409,0.0500,0.8851,0.1316,0.7403


## 5. Model final dan data uji (dipakai satu kali)
Model final = skor validasi TPR terbaik. Threshold diambil dari FPR 5% di validasi, lalu dipakai apa adanya di data uji.

In [7]:
final_name = val_table["TPR"].idxmax()
final_model = {"LogisticRegression": lr, "HistGradientBoosting (tuned)": hgb}[final_name]
threshold = scores[final_name]["threshold"]

test_score = evaluate(final_model, X_test, y_test, threshold=threshold)
print(f"Model final: {final_name} | threshold dari validasi: {threshold:.4f}")
print("Skor UJI (month 3):", {k: round(float(v), 4) for k, v in test_score.items()})

Model final: HistGradientBoosting (tuned) | threshold dari validasi: 0.7403
Skor UJI (month 3): {'TPR': 0.6932, 'FPR': 0.0882, 'ROC-AUC': 0.9113, 'PR-AUC': 0.1513, 'threshold': 0.7403}


## 6. Simpan, muat ulang, tes prediksi 5 baris

In [8]:
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"pipeline": final_model, "threshold": threshold}, MODEL_PATH)

saved = joblib.load(MODEL_PATH)
sample = X_test.sample(5, random_state=SEED)
proba = saved["pipeline"].predict_proba(sample)[:, 1]
pd.DataFrame({"fraud_asli": y_test[sample.index], "skor": proba.round(4),
              "ditandai_penipu": proba >= saved["threshold"]})

,fraud_asli,skor,ditandai_penipu
458796,0,0.1333,False
505674,0,0.0618,False
471909,0,0.0959,False
525013,0,0.0278,False
427917,0,0.8814,True
